# 03 — every trade against the book in force when it printed

DuckDB's `ASOF JOIN` pairs each trade with the last known BBO. `gold.bbo_1s` is the book at the **end** of its second, so the quote in force for a trade at `exchange_ts` is the row for the *previous* second — the join is on `second + 1 s <= exchange_ts`. Joining on `second <= exchange_ts` would use a quote from up to a second in the trade's future, and reads as 77 % of Binance prints trading through the book. From the correct pairing: inside, at, or through the quote, and the effective spread paid.

In [ ]:
from k2lake import connect, SCALE
con = connect()
DAY = con.sql("SELECT max(exchange_ts)::DATE FROM lake.gold.trades").fetchone()[0]
print('day', DAY)

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE t AS
SELECT exchange, canonical_symbol, exchange_ts, side, price_e8 / 100000000 AS price, qty_e8 / 100000000 AS qty
FROM lake.gold.trades WHERE exchange_ts::DATE = DATE '{DAY}' AND canonical_symbol IN ('BTC/USD', 'BTC/USDT');
CREATE OR REPLACE TEMP TABLE q AS
SELECT exchange, canonical_symbol, second, bid_e8 / 100000000 AS bid, ask_e8 / 100000000 AS ask, mid
FROM lake.gold.bbo_1s WHERE second::DATE = DATE '{DAY}' AND canonical_symbol IN ('BTC/USD', 'BTC/USDT');
""")
con.sql("SELECT (SELECT count(*) FROM t) AS trades, (SELECT count(*) FROM q) AS quotes").show()

In [ ]:
con.sql("""
CREATE OR REPLACE TEMP TABLE tq AS
SELECT t.*, q.bid, q.ask, q.mid, q.second AS quote_second
FROM t ASOF JOIN q ON t.exchange = q.exchange AND t.canonical_symbol = q.canonical_symbol AND t.exchange_ts >= q.second + INTERVAL 1 SECOND;
SELECT exchange, count(*) AS trades,
       round(100.0 * avg(CASE WHEN price BETWEEN bid AND ask THEN 1 ELSE 0 END), 2) AS pct_inside_or_at,
       round(100.0 * avg(CASE WHEN price > ask OR price < bid THEN 1 ELSE 0 END), 2) AS pct_through,
       round(avg(2 * abs(price - mid) / mid * 1e4), 3) AS effective_spread_bps,
       round(avg(CASE WHEN side = 'buy' THEN price - mid ELSE mid - price END) / avg(mid) * 1e4, 3) AS signed_cost_bps
FROM tq GROUP BY exchange ORDER BY exchange
""").show()

A trade `through` the quote is either a real sweep or a quote that was already stale: the BBO is between one and two seconds old by construction (end of the previous second). The split by age says how much of the through-rate is staleness.

In [ ]:
con.sql("""
SELECT exchange, CASE WHEN exchange_ts - quote_second < INTERVAL '1500' MILLISECOND THEN '1.0-1.5 s' ELSE '1.5-2 s+' END AS quote_age,
       count(*) AS trades, round(100.0 * avg(CASE WHEN price > ask OR price < bid THEN 1 ELSE 0 END), 2) AS pct_through
FROM tq GROUP BY 1, 2 ORDER BY 1, 2
""").show()